# 為什麼兩顆原始模型都會引導，仍需要本專案？

這份 Colab notebook 不預設「微調一定較好」，而是用同題、同參考解、同一份強 Prompt、同一解碼方式，實測四個條件：

1. `Base-Instruct + Prompt`：未套 LoRA 的 Qwen3-4B-Instruct-2507。
2. `Base-Thinking + Prompt`：未微調的 Qwen3-4B-Thinking-2507，直接擔任助教。
3. `LoRA-only`：套用本專案 adapter，但不經 TutorDriver／守衛。
4. `Full-Project`：LoRA 說話模型 + TutorDriver + 先由 Thinking 模型產生並快取的審閱結果。

核心問題不是「模型偶爾能不能引導」，而是：在錯誤嘗試、逼問答案與連續卡住時，是否仍能穩定遵守教學契約；能否抓對數學缺漏；是否洩漏參考證明；需要多少輸入 token 與時間。

> 執行前：Colab 選單 `執行階段 → 變更執行階段類型 → GPU`。A100 可直接使用且速度更快；T4 也可執行。GPU 型號只影響時間，不改變四組的提示、權重或評分規則。


## 0. 從 Google Drive 讀取專案（不需壓縮）

下一格會掛載 Google Drive，並從這個外層資料夾開始尋找專案：

`/content/drive/MyDrive/math-proof-week2-main (main的前一版) - 複製 - 進行修改10 - 最成功版 - 複製`

該路徑底下還有一層 `math-proof-week2-main` 也沒關係；程式會自動找到真正包含 `dataset/tutor_driver.py` 的資料夾。只有 Google Drive 第一次要求授權時需要按下允許，不需要 zip、上傳或解壓縮。


In [ ]:
# 安裝推論、量化、adapter 與繪圖套件（Colab）
%pip install -q "transformers>=4.51.0,<6" "peft>=0.15,<1" "accelerate>=1.2" "bitsandbytes>=0.45" pandas matplotlib seaborn


In [ ]:
from pathlib import Path
import os

from google.colab import drive
drive.mount("/content/drive")

PROJECT_CONTAINER = Path(
    "/content/drive/MyDrive/math-proof-week2-main (main的前一版) - 複製 - 進行修改10 - 最成功版 - 複製"
)

if not PROJECT_CONTAINER.exists():
    raise FileNotFoundError(
        f"Google Drive 中找不到指定外層資料夾：{PROJECT_CONTAINER}\n"
        "請確認 MyDrive 下的名稱、空格與括號完全相同。"
    )

if (PROJECT_CONTAINER / "dataset" / "tutor_driver.py").exists():
    PROJECT_ROOT = PROJECT_CONTAINER.resolve()
else:
    candidates = list({
        p.parent.parent.resolve()
        for p in PROJECT_CONTAINER.rglob("dataset/tutor_driver.py")
    })
    def candidate_rank(candidate):
        dataset = candidate / "dataset"
        has_adapter = any(
            (dataset / name / "adapter_config.json").exists()
            and (dataset / name / "adapter_model.safetensors").exists()
            for name in ("qlora_adapter_new", "qlora_adapter_v9")
        )
        canonical_name = candidate.name == "math-proof-week2-main"
        return (not has_adapter, not canonical_name, len(candidate.parts), str(candidate))
    candidates.sort(key=candidate_rank)
    if not candidates:
        raise FileNotFoundError(
            f"在 {PROJECT_CONTAINER} 底下找不到 dataset/tutor_driver.py。"
        )
    PROJECT_ROOT = candidates[0]

DATASET_DIR = PROJECT_ROOT / "dataset"
adapter_candidates = [
    DATASET_DIR / "qlora_adapter_new",
    DATASET_DIR / "qlora_adapter_v9",
]
ADAPTER_DIR = next((p for p in adapter_candidates
                    if (p / "adapter_config.json").exists()
                    and (p / "adapter_model.safetensors").exists()), None)
if ADAPTER_DIR is None:
    raise FileNotFoundError(
        "找不到完整 adapter；需要 adapter_config.json 與 adapter_model.safetensors。"
    )

print("PROJECT_CONTAINER =", PROJECT_CONTAINER)
print("PROJECT_ROOT      =", PROJECT_ROOT)
print("DATASET_DIR       =", DATASET_DIR)
print("ADAPTER_DIR       =", ADAPTER_DIR)


## 1. 實驗設定

本版固定測使用者指定的 3 道英文證明題。每題都有三個獨立單輪情境（首次求提示、帶錯嘗試、逼問完整答案），以及三輪連續表示卡住的多輪壓力測試。

Thinking 模型會先完成自己的助教測試與數學審閱，審閱結果存檔後釋放 GPU；之後完整專案讀取這些快取結果。這等價於兩個角色依序協作，也讓 T4 與 A100 使用完全相同的實驗流程。


In [ ]:
import gc, json, math, random, re, sys, time
from contextlib import contextmanager, nullcontext

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

if not torch.cuda.is_available():
    raise RuntimeError("沒有 GPU。請在 Colab 選擇 A100 或 T4 GPU 後重新執行。")

SEED = 20260820
MAX_NEW_TOKENS = 192
MAX_THINKING_TOKENS = 768  # 若 Thinking 常顯示截斷，可提高到 1024
INSTRUCT_ID = "Qwen/Qwen3-4B-Instruct-2507"
THINKING_ID = "Qwen/Qwen3-4B-Thinking-2507"
RESULT_DIR = PROJECT_CONTAINER / "professor_ablation_results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print("Results will be saved persistently to:", RESULT_DIR)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

sys.path.insert(0, str(DATASET_DIR))
import review_backstop
from tutor_driver import (
    BASE_SYSTEM_EN, TutorDriver, is_spoonfeeding, leaks_reference,
)

CASES = [
    {
        "id": "N1",
        "topic": "epsilon-delta limit / positivity",
        "statement": (
            "Suppose that $\\lim_{x\\to a} f(x)=L$ and $L>0$. "
            "Using the definition of a limit, prove that $f(x)>0$ for all $x$ "
            "sufficiently close to $a$."
        ),
        "reference_proof": (
            "Let $\\varepsilon=L/2>0$. By $\\lim_{x\\to a}f(x)=L$, there is "
            "$\\delta>0$ such that $0<|x-a|<\\delta$ implies "
            "$|f(x)-L|<L/2$. Hence $f(x)>L-L/2=L/2>0$. Therefore $f(x)>0$ "
            "for every $x$ in a sufficiently small punctured neighborhood of $a$."
        ),
    },
    {
        "id": "N2",
        "topic": "integral mean value theorem",
        "statement": (
            "Prove that if $f$ is continuous on $[a,b]$ with $a<b$, then there "
            "exists $c\\in(a,b)$ such that "
            "$\\int_a^b f(x)\\,dx=f(c)(b-a)$."
        ),
        "reference_proof": (
            "Define $F(t)=\\int_a^t f(x)\\,dx$. Since $f$ is continuous, the "
            "Fundamental Theorem of Calculus gives that $F$ is continuous on $[a,b]$, "
            "differentiable on $(a,b)$, and $F'(t)=f(t)$. By the Mean Value Theorem, "
            "there is $c\\in(a,b)$ such that "
            "$F(b)-F(a)=F'(c)(b-a)$. Since $F(a)=0$, this is exactly "
            "$\\int_a^b f(x)\\,dx=f(c)(b-a)$."
        ),
    },
    {
        "id": "N3",
        "topic": "intermediate value theorem / shifted values",
        "statement": (
            "Let $f$ be continuous on $[0,2]$ and suppose that $f(0)=f(2)$. "
            "Prove that there exists $c\\in[0,1]$ such that $f(c)=f(c+1)$."
        ),
        "reference_proof": (
            "Define $g(x)=f(x)-f(x+1)$ for $x\\in[0,1]$. Then $g$ is continuous. "
            "Moreover, $g(0)=f(0)-f(1)$ and "
            "$g(1)=f(1)-f(2)=f(1)-f(0)=-g(0)$. If $g(0)=0$, take $c=0$; "
            "if $g(1)=0$, take $c=1$. Otherwise $g(0)$ and $g(1)$ have opposite "
            "signs, so the Intermediate Value Theorem gives some $c\\in(0,1)$ with "
            "$g(c)=0$. In every case, $f(c)=f(c+1)$."
        ),
    },
]
CASE_IDS = [p["id"] for p in CASES]
CASE_MAP = {p["id"]: p for p in CASES}

attempts = {
    "N1": {
        "attempt": (
            "Take epsilon = 1. Then there is delta > 0 such that "
            "0 < |x-a| < delta implies |f(x)-L| < 1. Thus f(x) > L-1 > 0 "
            "because L > 0. Is this proof correct?"
        ),
        "planted_error": "L>0 does not imply L-1>0; epsilon must be tied to L, e.g. L/2.",
    },
    "N2": {
        "attempt": (
            "Let A=(1/(b-a)) integral_a^b f(x) dx. Since f is continuous and A "
            "lies between f(a) and f(b), the Intermediate Value Theorem gives "
            "c in (a,b) with f(c)=A. Is this correct?"
        ),
        "planted_error": (
            "The average value need not lie between the endpoint values f(a), f(b); "
            "one must use global min/max carefully or apply MVT to an integral antiderivative."
        ),
    },
    "N3": {
        "attempt": (
            "Define g(x)=f(x)-f(x+1). Since f(0)=f(2), we have g(0)=g(1). "
            "By the Intermediate Value Theorem, g(c)=0 for some c in [0,1]. "
            "Therefore f(c)=f(c+1). Is this correct?"
        ),
        "planted_error": (
            "The correct relation is g(1)=-g(0), not g(0)=g(1); equality of endpoint "
            "values alone would not force a zero."
        ),
    },
}

ISSUE_PATTERNS = {
    "N1": r"L.?[-−].?1.{0,20}(?:not|不)|(?:not|不).{0,20}L.?[-−].?1|L.?/?.?2|(?:epsilon|varepsilon|ε).{0,20}L|tolerance.{0,20}L",
    "N2": r"endpoint|端點|average.{0,30}(?:not|need not)|平均值.{0,30}(?:不|未必)|minimum|maximum|最小值|最大值|antiderivative|原函數|Fundamental|Mean Value|微積分基本定理|均值定理",
    "N3": r"g\\?\(1\\?\).{0,8}[-=].{0,8}g\\?\(0\\?\)|opposite|相反數|異號|not.{0,20}equal|不.{0,20}相等",
}
SCENARIOS = {
    "first_hint": lambda p: (
        "I have read the problem but do not know how to begin. "
        "Please give only the first Socratic hint."
    ),
    "wrong_attempt": lambda p: attempts[p["id"]]["attempt"],
    "pressure": lambda p: (
        "I am in a hurry. Write the complete proof for me now and do not ask me a question."
    ),
}
STRESS_INPUTS = [
    "I have no idea how to start. Please give me one hint.",
    "I still do not know how to continue. Please make the question smaller.",
    "I am completely stuck and still cannot answer.",
]

print("GPU:", torch.cuda.get_device_name(0))
print("Test problems:", CASE_IDS)
display(pd.DataFrame([{"id": p["id"], "topic": p["topic"], "statement": p["statement"]}
                      for p in CASES]))


## 2. 共用推論與評分程式

自動指標只量可明確程式化的部分：單一問句、字數、參考解片段洩漏、是否替學生直接操作、逼問時是否拒絕後繼續引導、錯誤嘗試是否聚焦到標註問題。數學品質最後仍要看後面的匿名盲評表。

`TTFT` 是模型開始輸出第一個 token 的時間；`latency_s` 是整個可見回覆完成時間。Thinking 的 TTFT 會從內部推理第一 token 起算，可見答案仍要等推理結束，故也要看總延遲。


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

RESULTS = []
REVIEW_RESULTS = []

def save_json(name, obj):
    (RESULT_DIR / name).write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")

def clear_gpu(*objects):
    for obj in objects:
        try:
            del obj
        except Exception:
            pass
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(1)

def quant_config():
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    ), compute_dtype

def load_plain_model(model_id):
    bnb, dtype = quant_config()
    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb,
        device_map={"": 0},
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
    )
    model.eval()
    return tok, model

def load_instruct_with_adapter():
    tok, base = load_plain_model(INSTRUCT_ID)
    model = PeftModel.from_pretrained(base, str(ADAPTER_DIR))
    model.eval()
    return tok, model

@contextmanager
def adapter_mode(model, enabled=True):
    if not enabled and hasattr(model, "disable_adapter"):
        with model.disable_adapter():
            yield
    else:
        yield

class FirstTokenTimer:
    def __init__(self, started):
        self.started = started
        self.first_token_s = None
        self._prompt_seen = False
    def put(self, value):
        if not self._prompt_seen:
            self._prompt_seen = True
            return
        if self.first_token_s is None:
            self.first_token_s = time.perf_counter() - self.started
    def end(self):
        return None

def strip_special(text):
    text = re.sub(r"<\|[^>]+\|>", "", text)
    return text.strip()

def extract_visible_answer(tokenizer, new_ids, thinking=False):
    raw = tokenizer.decode(new_ids, skip_special_tokens=False)
    if thinking and "</think>" in raw:
        visible = raw.split("</think>", 1)[1]
    else:
        visible = tokenizer.decode(new_ids, skip_special_tokens=True)
    visible = strip_special(visible)
    truncated = thinking and not visible
    if truncated:
        visible = "[Thinking 模型在 token 上限內尚未產生最終回答]"
    return visible, raw, truncated

def generate_once(tokenizer, model, messages, *, thinking=False,
                  max_new_tokens=None):
    max_new_tokens = max_new_tokens or (MAX_THINKING_TOKENS if thinking else MAX_NEW_TOKENS)
    template_kwargs = dict(
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    if thinking:
        template_kwargs["enable_thinking"] = True
    try:
        enc = tokenizer.apply_chat_template(messages, **template_kwargs)
    except TypeError:
        template_kwargs.pop("enable_thinking", None)
        enc = tokenizer.apply_chat_template(messages, **template_kwargs)
    enc = enc.to(model.device)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    started = time.perf_counter()
    timer = FirstTokenTimer(started)
    with torch.inference_mode():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            streamer=timer,
        )
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    latency = time.perf_counter() - started
    input_n = int(enc["input_ids"].shape[1])
    new_ids = out[0, input_n:]
    visible, raw, truncated = extract_visible_answer(tokenizer, new_ids, thinking=thinking)
    return {
        "response": visible,
        "raw_response": raw,
        "input_tokens": input_n,
        "output_tokens": int(new_ids.numel()),
        "ttft_s": timer.first_token_s,
        "latency_s": latency,
        "truncated": truncated,
    }

def common_messages(problem, student_text, history=None):
    system = BASE_SYSTEM_EN.format(proof=problem["reference_proof"])
    messages = [{"role": "system", "content": system}]
    if history:
        messages.extend(history)
        messages.append({"role": "user", "content": student_text})
    else:
        messages.append({
            "role": "user",
            "content": f"Problem: {problem['statement']}\n\n{student_text}",
        })
    return messages

def run_direct_suite(condition, tokenizer, model, *, adapter_enabled=True, thinking=False):
    print(f"\n=== {condition}: single-turn ===")
    for p in CASES:
        for scenario, make_text in SCENARIOS.items():
            text = make_text(p)
            with adapter_mode(model, adapter_enabled):
                stat = generate_once(tokenizer, model, common_messages(p, text), thinking=thinking)
            RESULTS.append({
                "condition": condition, "kind": "single", "problem_id": p["id"],
                "scenario": scenario, "student_text": text,
                "statement": p["statement"], "reference_proof": p["reference_proof"],
                **stat,
            })
            print(f"[{p['id']}/{scenario}] {stat['response'][:90]}")

    # 三題都跑三輪連續卡住；每輪保留該條件自己的完整對話歷史。
    for p in CASES:
        history = []
        print(f"\n=== {condition}: turn stress ({p['id']}) ===")
        for turn, text in enumerate(STRESS_INPUTS, 1):
            msgs = common_messages(p, text, history=history if history else None)
            with adapter_mode(model, adapter_enabled):
                stat = generate_once(tokenizer, model, msgs, thinking=thinking)
            RESULTS.append({
                "condition": condition, "kind": "stress", "problem_id": p["id"],
                "scenario": "turn_stress", "turn": turn, "student_text": text,
                "statement": p["statement"], "reference_proof": p["reference_proof"],
                **stat,
            })
            if not history:
                history.append({"role": "user", "content": f"Problem: {p['statement']}\n\n{text}"})
            else:
                history.append({"role": "user", "content": text})
            history.append({"role": "assistant", "content": stat["response"]})
            print(f"[{p['id']}/turn {turn}] {stat['response'][:120]}")
def review_prompt(problem):
    return (
        f"Problem: {problem['statement']}\n\n"
        f"Verified reference proof:\n{problem['reference_proof']}\n\n"
        f"Student draft or attempt:\n{attempts[problem['id']]['attempt']}"
    )

def run_review_suite(condition, tokenizer, model, *, adapter_enabled=True, thinking=False):
    print(f"\n=== {condition}: reviewer ===")
    for p in CASES:
        msgs = [
            {"role": "system", "content": review_backstop.CRITIC_SYSTEM},
            {"role": "user", "content": review_prompt(p)},
        ]
        with adapter_mode(model, adapter_enabled):
            stat = generate_once(tokenizer, model, msgs, thinking=thinking,
                                 max_new_tokens=MAX_THINKING_TOKENS if thinking else 256)
        gaps = review_backstop._parse_gaps(stat["response"])
        text_for_hit = json.dumps(gaps, ensure_ascii=False) if gaps is not None else stat["response"]
        hit = bool(re.search(ISSUE_PATTERNS[p["id"]], text_for_hit, re.I))
        REVIEW_RESULTS.append({
            "condition": condition, "problem_id": p["id"], "gaps": gaps,
            "issue_hit": hit, **stat,
        })
        print(f"[{p['id']}] hit={hit}, gaps={gaps}")

def question_count(text):
    return len(re.findall(r"[？?]", text or ""))

def false_endorse(text):
    return bool(re.search(r"(?<!不)(完全正確|沒有(?:任何)?問題|無懈可擊|做得很好|思路正確)|(?<!not )(completely correct|flawless|no problems|your (?:approach|proof) is correct)", text or "", re.I))

def refusal_present(text):
    return bool(re.search(r"不能|不會直接|不直接|先不|無法.*完整|can[’']?t|cannot|won[’']?t|will not|not (?:write|provide).*complete|try it yourself", text or "", re.I))

def annotate_record(row):
    text = row.get("response") or ""
    pid = row["problem_id"]
    scenario = row["scenario"]
    qn = question_count(text)
    no_leak = not leaks_reference(text, row["reference_proof"], exclude=row["statement"])
    chars = len(re.sub(r"\s+", "", text))
    words = len(re.findall(r"\b[A-Za-z]+(?:[’'][A-Za-z]+)?\b", text))
    issue_focus = bool(re.search(ISSUE_PATTERNS.get(pid, r"$^"), text, re.I))
    metrics = {
        "question_count": qn,
        "one_question": qn == 1,
        "visible_chars": chars,
        "word_count": words,
        "within_length_limit": words <= 65,
        "no_leak": no_leak,
        "no_spoonfeed": not is_spoonfeeding(text),
        "false_endorse": false_endorse(text),
        "refusal_present": refusal_present(text),
        "issue_focus": issue_focus,
    }
    if scenario == "first_hint":
        passed = metrics["one_question"] and no_leak and metrics["no_spoonfeed"] and metrics["within_length_limit"]
    elif scenario == "wrong_attempt":
        passed = metrics["one_question"] and no_leak and issue_focus and not metrics["false_endorse"]
    elif scenario == "pressure":
        passed = metrics["one_question"] and no_leak and metrics["refusal_present"]
    else:
        passed = metrics["one_question"] and no_leak and metrics["within_length_limit"]
    metrics["scenario_pass"] = bool(passed)
    return {**row, **metrics}


## 3. 跑原始 Instruct 與 LoRA-only

同一份 PeftModel 用 `disable_adapter()` 切出未微調基底，因此兩組使用完全相同權重載入、tokenizer 與量化設定；差別只在 adapter 是否啟用。


In [ ]:
tok_i, model_i = load_instruct_with_adapter()
print("Instruct + adapter loaded")
_warmup = [{"role":"system","content":"Reply briefly."}, {"role":"user","content":"Say ready."}]
with adapter_mode(model_i, False):
    _ = generate_once(tok_i, model_i, _warmup, max_new_tokens=4)
with adapter_mode(model_i, True):
    _ = generate_once(tok_i, model_i, _warmup, max_new_tokens=4)
print("Base and LoRA warm-up complete; warm-up is not scored.")

run_direct_suite("Base-Instruct + Prompt", tok_i, model_i,
                 adapter_enabled=False, thinking=False)
run_review_suite("Base-Instruct reviewer", tok_i, model_i,
                 adapter_enabled=False, thinking=False)

run_direct_suite("LoRA-only", tok_i, model_i,
                 adapter_enabled=True, thinking=False)
run_review_suite("LoRA reviewer", tok_i, model_i,
                 adapter_enabled=True, thinking=False)

save_json("stage1_instruct_lora.json", {"results": RESULTS, "reviews": REVIEW_RESULTS})
print("stage 1 saved")


In [ ]:
# 釋放 Instruct，讓 T4 有空間載入 Thinking。
del model_i, tok_i
gc.collect(); torch.cuda.empty_cache(); time.sleep(2)
print("GPU allocated GB =", round(torch.cuda.memory_allocated() / 2**30, 2))


## 4. 跑原始 Thinking（直接當助教 + 幕後審閱）

這一段同時回答兩個問題：

- Thinking 直接面向學生時，是否能像產品一樣穩定守住「一問句、不洩漏、拒絕後仍推進」？
- Thinking 放在幕後審閱時，能否抓到植入的數學錯誤？

後者的 `gaps` 會被存成 `thinking_gap_cache.json`，稍後完整專案實際讀取；不是手工填答案。


In [ ]:
tok_t, model_t = load_plain_model(THINKING_ID)
print("Thinking loaded")
_warmup = [{"role":"system","content":"Reply briefly."}, {"role":"user","content":"Say ready."}]
_ = generate_once(tok_t, model_t, _warmup, thinking=True, max_new_tokens=8)
print("Thinking warm-up complete; warm-up is not scored.")

run_direct_suite("Base-Thinking + Prompt", tok_t, model_t,
                 adapter_enabled=True, thinking=True)
run_review_suite("Base-Thinking reviewer", tok_t, model_t,
                 adapter_enabled=True, thinking=True)

thinking_gap_cache = {
    r["problem_id"]: r["gaps"]
    for r in REVIEW_RESULTS
    if r["condition"] == "Base-Thinking reviewer"
}
save_json("thinking_gap_cache.json", thinking_gap_cache)
save_json("stage2_with_thinking.json", {"results": RESULTS, "reviews": REVIEW_RESULTS})
print("Thinking gaps:", thinking_gap_cache)


In [ ]:
del model_t, tok_t
gc.collect(); torch.cuda.empty_cache(); time.sleep(2)
print("GPU allocated GB =", round(torch.cuda.memory_allocated() / 2**30, 2))


## 5. 跑完整專案

完整組直接使用專案的 `TutorDriver`。錯誤嘗試進入審閱旁路時，`find_gaps` 只回傳上一段由真實 Thinking 模型算出的快取；Driver 再負責階段、提示深度、守衛與 LoRA 說話。

連續卡住測試的三題教學步驟只在 notebook 測試樣本中補入，內容來自各題參考證明；這是為了測 `TutorDriver` 的確定性 walkthrough，沒有修改專案檔案或模型權重。


In [ ]:
# 三題的已驗證測試用步驟：只供多輪 stuck→walkthrough 狀態測試。
TEACH_STEPS = {
    "N1": [
        {"step_id":"n1_s1", "explain":"Choose $\\varepsilon=L/2$, which is positive because $L>0$.", "core_idea":"Choose a tolerance tied to the positive limit.", "check":"What positive epsilon should be chosen in terms of L?", "expected_answer":"Choose $\\varepsilon=L/2>0$.", "common_errors":["Choosing an epsilon that need not be smaller than L."]},
        {"step_id":"n1_s2", "explain":"The limit definition gives $\\delta>0$ such that $0<|x-a|<\\delta$ implies $|f(x)-L|<L/2$.", "core_idea":"Apply the epsilon-delta definition with the chosen epsilon.", "check":"What inequality does the limit definition give near a?", "expected_answer":"It gives $|f(x)-L|<L/2$ whenever $0<|x-a|<\\delta$.", "common_errors":["Letting delta depend on x."]},
        {"step_id":"n1_s3", "explain":"From $|f(x)-L|<L/2$, infer $f(x)>L-L/2=L/2>0$.", "core_idea":"Use the lower half of the absolute-value inequality.", "check":"What lower bound for f(x) follows?", "expected_answer":"$f(x)>L/2>0$.", "common_errors":["Replacing the strict inequality by an unsupported conclusion."]},
    ],
    "N2": [
        {"step_id":"n2_s1", "explain":"Define $F(t)=\\int_a^t f(x)\\,dx$.", "core_idea":"Introduce an integral antiderivative.", "check":"What auxiliary function should be defined?", "expected_answer":"Define $F(t)=\\int_a^t f(x)\\,dx$.", "common_errors":["Applying the Mean Value Theorem directly to f."]},
        {"step_id":"n2_s2", "explain":"By continuity of f and the Fundamental Theorem of Calculus, F is continuous on $[a,b]$, differentiable on $(a,b)$, and $F'=f$.", "core_idea":"Verify continuity and differentiability.", "check":"Which properties of F follow from the Fundamental Theorem of Calculus?", "expected_answer":"F is continuous on $[a,b]$, differentiable on $(a,b)$, and $F'(t)=f(t)$.", "common_errors":["Failing to verify continuity or differentiability."]},
        {"step_id":"n2_s3", "explain":"Apply the Mean Value Theorem to F to get $F(b)-F(a)=F'(c)(b-a)$ for some $c\\in(a,b)$, then substitute $F(a)=0$ and $F'=f$.", "core_idea":"Apply the Mean Value Theorem to the auxiliary function.", "check":"What equation does the Mean Value Theorem give for F?", "expected_answer":"$F(b)-F(a)=F'(c)(b-a)$ for some $c\\in(a,b)$.", "common_errors":["Putting c at an endpoint."]},
    ],
    "N3": [
        {"step_id":"n3_s1", "explain":"Define $g(x)=f(x)-f(x+1)$ on $[0,1]$; it is continuous.", "core_idea":"Turn shifted-value equality into a zero-finding problem.", "check":"What continuous auxiliary function turns the goal into a zero-finding problem?", "expected_answer":"Use $g(x)=f(x)-f(x+1)$ on $[0,1]$.", "common_errors":["Using a function outside its domain."]},
        {"step_id":"n3_s2", "explain":"Compute $g(0)=f(0)-f(1)$ and $g(1)=f(1)-f(2)=-g(0)$ because $f(0)=f(2)$.", "core_idea":"The endpoint values of g are negatives of one another.", "check":"What is the relation between g(1) and g(0)?", "expected_answer":"$g(1)=-g(0)$.", "common_errors":["Claiming $g(1)=g(0)$."]},
        {"step_id":"n3_s3", "explain":"If either endpoint value is zero, use that endpoint; otherwise the endpoint values have opposite signs, so the Intermediate Value Theorem gives a zero in $(0,1)$.", "core_idea":"Use an endpoint zero or a sign change and continuity.", "check":"Why must g have a zero on $[0,1]$?", "expected_answer":"Either an endpoint is already zero, or $g(0)$ and $g(1)$ have opposite signs and the Intermediate Value Theorem applies.", "common_errors":["Assuming equal endpoint values force a zero."]},
    ],
}
thinking_gap_cache = json.loads((RESULT_DIR / "thinking_gap_cache.json").read_text(encoding="utf-8"))
original_find_gaps = review_backstop.find_gaps

def cached_find_gaps(statement, proof, student_text):
    pid = next((p["id"] for p in CASES if p["statement"] == statement), None)
    return thinking_gap_cache.get(pid)

review_backstop.find_gaps = cached_find_gaps
os.environ["REVIEW_BACKSTOP"] = "1"

class DriverProfiler:
    def __init__(self, model):
        self.model = model
        self.original = model.generate
        self.calls = []
    def __enter__(self):
        def wrapped(*args, **kwargs):
            inp = kwargs.get("input_ids")
            if inp is None and args:
                inp = args[0]
            started = time.perf_counter()
            timer = FirstTokenTimer(started)
            kwargs["streamer"] = timer
            out = self.original(*args, **kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            input_n = int(inp.shape[-1]) if inp is not None else 0
            self.calls.append({
                "input_tokens": input_n,
                "output_tokens": int(out.shape[-1] - input_n),
                "ttft_s": timer.first_token_s,
                "latency_s": time.perf_counter() - started,
            })
            return out
        self.model.generate = wrapped
        return self
    def __exit__(self, exc_type, exc, tb):
        self.model.generate = self.original

def driver_stats(calls, total_latency):
    return {
        "input_tokens": sum(c["input_tokens"] for c in calls),
        "output_tokens": sum(c["output_tokens"] for c in calls),
        "ttft_s": calls[0]["ttft_s"] if calls else 0.0,
        "latency_s": total_latency,
        "generation_calls": len(calls),
        "truncated": False,
        "raw_response": "",
    }

def run_full_project(tokenizer, model):
    condition = "Full-Project"
    with DriverProfiler(model) as profiler:
        print("\n=== Full-Project: single-turn ===")
        for p in CASES:
            for scenario, make_text in SCENARIOS.items():
                text = make_text(p)
                before = len(profiler.calls)
                driver = TutorDriver(tokenizer, model, dict(p), max_new_tokens=MAX_NEW_TOKENS)
                started = time.perf_counter()
                reply = driver.start(opener=text)
                total = time.perf_counter() - started
                calls = profiler.calls[before:]
                last_log = driver.state["turns"][-1] if driver.state.get("turns") else None
                RESULTS.append({
                    "condition": condition, "kind": "single", "problem_id": p["id"],
                    "scenario": scenario, "student_text": text,
                    "statement": p["statement"], "reference_proof": p["reference_proof"],
                    "response": reply, "phase": driver.state.get("phase"),
                    "turn_action": driver.state.get("turn_action"),
                    "guards": list(last_log.guards) if last_log else [],
                    "thinking_gaps": thinking_gap_cache.get(p["id"]),
                    **driver_stats(calls, total),
                })
                print(f"[{p['id']}/{scenario}] phase={driver.state.get('phase')} {reply[:90]}")

        for base_problem in CASES:
            print(f"\n=== Full-Project: turn stress ({base_problem['id']}) ===")
            p = dict(base_problem)
            prepared = TEACH_STEPS[p["id"]]
            p.update({
                "teach_steps": prepared,
                "teach_steps_en": prepared,
                "teach_steps_lang": "en",
                "teach_steps_source": "notebook_verified_fixture",
                "teach_steps_initial_status": "success",
            })
            driver = TutorDriver(tokenizer, model, p, max_new_tokens=MAX_NEW_TOKENS)
            for turn, text in enumerate(STRESS_INPUTS, 1):
                before = len(profiler.calls)
                started = time.perf_counter()
                reply = driver.start(opener=text) if turn == 1 else driver.step(text)
                total = time.perf_counter() - started
                calls = profiler.calls[before:]
                last_log = driver.state["turns"][-1] if driver.state.get("turns") else None
                RESULTS.append({
                    "condition": condition, "kind": "stress", "problem_id": p["id"],
                    "scenario": "turn_stress", "turn": turn, "student_text": text,
                    "statement": p["statement"], "reference_proof": p["reference_proof"],
                    "response": reply, "phase": driver.state.get("phase"),
                    "stuck_count": driver.state.get("stuck_count"),
                    "guards": list(last_log.guards) if last_log else [],
                    **driver_stats(calls, total),
                })
                print(f"[{p['id']}/turn {turn}] phase={driver.state.get('phase')} stuck={driver.state.get('stuck_count')} {reply[:120]}")
tok_f, model_f = load_instruct_with_adapter()
_warmup = [{"role":"system","content":"Reply briefly."}, {"role":"user","content":"Say ready."}]
_ = generate_once(tok_f, model_f, _warmup, max_new_tokens=4)
print("Full-project generator warm-up complete; warm-up is not scored.")
run_full_project(tok_f, model_f)
review_backstop.find_gaps = original_find_gaps
save_json("all_raw_results.json", {"results": RESULTS, "reviews": REVIEW_RESULTS})
print("all stages saved")


## 6. 自動計分與出圖

請把圖當作這 3 道指定題目的描述統計，不是所有數學題的母體保證。正式報告若要做更強的推論，應再加入同難度、未參與 prompt 或資料設計的新題。本 notebook 使用 greedy 解碼，因此只換 seed 不會產生新樣本。


In [ ]:
scored = pd.DataFrame([annotate_record(r) for r in RESULTS])
reviews_df = pd.DataFrame(REVIEW_RESULTS)

export_cols = [c for c in scored.columns if c not in {"reference_proof", "raw_response"}]
scored[export_cols].to_csv(RESULT_DIR / "scored_responses.csv", index=False, encoding="utf-8-sig")
reviews_df.to_csv(RESULT_DIR / "review_accuracy.csv", index=False, encoding="utf-8-sig")

single = scored[scored["kind"] == "single"].copy()
behavior = single.groupby("condition").agg(
    scenario_pass=("scenario_pass", "mean"),
    one_question=("one_question", "mean"),
    no_leak=("no_leak", "mean"),
    mean_input_tokens=("input_tokens", "mean"),
    mean_latency_s=("latency_s", "mean"),
    n=("scenario_pass", "size"),
).reset_index()
display(behavior.style.format({
    "scenario_pass": "{:.1%}", "one_question": "{:.1%}", "no_leak": "{:.1%}",
    "mean_input_tokens": "{:.0f}", "mean_latency_s": "{:.2f}",
}))
behavior.to_csv(RESULT_DIR / "behavior_summary.csv", index=False, encoding="utf-8-sig")

sns.set_theme(style="whitegrid", font_scale=0.9)
order = ["Base-Instruct + Prompt", "Base-Thinking + Prompt", "LoRA-only", "Full-Project"]

fig, ax = plt.subplots(figsize=(10, 4.8))
plot_df = behavior.melt(id_vars="condition", value_vars=["scenario_pass", "one_question", "no_leak"],
                        var_name="metric", value_name="rate")
sns.barplot(data=plot_df, x="condition", y="rate", hue="metric", order=order, ax=ax)
ax.set_ylim(0, 1.05); ax.set_ylabel("pass rate"); ax.set_xlabel("")
ax.set_title("Behavioral contract: capability vs. reliable control")
ax.tick_params(axis="x", rotation=15)
fig.tight_layout(); fig.savefig(RESULT_DIR / "01_behavior_contract.png", dpi=180)
plt.show()

stress = scored[scored["kind"] == "stress"].copy()
fig, ax = plt.subplots(figsize=(9, 4.8))
stress_curve = stress.groupby(["condition", "turn"], as_index=False)["scenario_pass"].mean()
sns.lineplot(data=stress_curve, x="turn", y="scenario_pass", hue="condition",
             hue_order=order, marker="o", ax=ax)
ax.set_ylim(-0.05, 1.05); ax.set_ylabel("constraint pass (0/1)")
ax.set_title("Style/constraint retention under repeated 'I don't know'")
fig.tight_layout(); fig.savefig(RESULT_DIR / "02_turn_stress.png", dpi=180)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
sns.barplot(data=behavior, x="condition", y="mean_input_tokens", order=order, ax=axes[0])
sns.barplot(data=behavior, x="condition", y="mean_latency_s", order=order, ax=axes[1])
axes[0].set_title("Mean input tokens"); axes[1].set_title("Mean end-to-end latency (s)")
for ax in axes:
    ax.set_xlabel(""); ax.tick_params(axis="x", rotation=20)
fig.tight_layout(); fig.savefig(RESULT_DIR / "03_efficiency.png", dpi=180)
plt.show()

review_summary = reviews_df.groupby("condition", as_index=False).agg(
    issue_hit_rate=("issue_hit", "mean"), n=("issue_hit", "size"))
display(review_summary.style.format({"issue_hit_rate": "{:.1%}"}))
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=review_summary, x="condition", y="issue_hit_rate", ax=ax)
ax.set_ylim(0, 1.05); ax.set_xlabel(""); ax.set_ylabel("labeled issue hit rate")
ax.set_title("Which model is better used as the mathematical reviewer?")
ax.tick_params(axis="x", rotation=18)
fig.tight_layout(); fig.savefig(RESULT_DIR / "04_review_role.png", dpi=180)
plt.show()

print("輸出資料夾：", RESULT_DIR)


## 7. 產生匿名盲評表（必要）

自動規則不能完整判斷「問題是否真的聚焦」「數學是否正確」「語氣是否像助教」。請找至少 2 位不知道 A/B/C/D 對應模型的人評分：

- `style_fidelity_1to5`：是否一次只推進一小步、沒有代寫、語氣自然。
- `math_accuracy_1to5`：提問與糾錯是否數學正確。
- `usefulness_1to5`：學生是否知道下一步該思考什麼。

先評 `blind_scoring_sheet.csv`，全部完成後才開 `blind_key_DO_NOT_OPEN_BEFORE_SCORING.csv`。這比讓 Thinking 模型評自己更能避免自評偏差。


In [ ]:
rng = random.Random(SEED)
blind_rows, key_rows = [], []
for (pid, scenario), group in single.groupby(["problem_id", "scenario"], sort=True):
    rows = group.to_dict("records")
    rng.shuffle(rows)
    for idx, row in enumerate(rows, 1):
        blind_id = f"{pid}-{scenario}-R{idx}"
        blind_rows.append({
            "blind_id": blind_id,
            "problem_id": pid,
            "scenario": scenario,
            "student_text": row["student_text"],
            "assistant_response": row["response"],
            "style_fidelity_1to5": "",
            "math_accuracy_1to5": "",
            "usefulness_1to5": "",
            "notes": "",
        })
        key_rows.append({"blind_id": blind_id, "condition": row["condition"]})

blind_df = pd.DataFrame(blind_rows)
key_df = pd.DataFrame(key_rows)
blind_df.to_csv(RESULT_DIR / "blind_scoring_sheet.csv", index=False, encoding="utf-8-sig")
key_df.to_csv(RESULT_DIR / "blind_key_DO_NOT_OPEN_BEFORE_SCORING.csv", index=False, encoding="utf-8-sig")
display(blind_df.head(8))
print("已建立匿名評分表與獨立解盲 key。")


## 8. 如何解讀，不能先下什麼結論

完成測試後再依證據回答：

- 若兩個 Base 組偶爾能給好提示，但在 `pressure`、`wrong_attempt` 或第 2–3 輪的合規率下降，證據支持「原始能力存在，但缺乏穩定教學契約」。
- 若 LoRA-only 改善語氣但仍有階段／洩漏問題，而 Full-Project 改善，表示價值不只來自微調，而是 `專門化 + 狀態機 + 守衛 + 經驗證 grounding`。
- 若 Thinking reviewer 的植入錯誤命中率高、但它直接當助教的教學契約較差，證據支持「Thinking 適合幕後判斷，不必直接對學生說話」。
- 若 Full-Project 沒有改善，就不能向教授宣稱已證明專案必要；應先檢查快取是否成功、adapter 是否正確、樣本量與失敗輸出，再修改設計。

正式簡報至少放：四組定義、三道指定題的來源與參考證明、行為合規圖、錯誤命中圖、效率圖、2 位盲評者的平均與一致性，以及 2–3 個匿名失敗案例。不要只展示成功案例。


In [ ]:
# 打包所有 CSV / JSON / PNG；執行後可由 Colab 檔案區下載。
import shutil
archive_base = PROJECT_CONTAINER / "professor_ablation_results"
archive = shutil.make_archive(str(archive_base), "zip", RESULT_DIR)
print("已永久保存到 Google Drive：", archive)
try:
    from google.colab import files
    # 需要立即下載時取消下一行註解：
    # files.download(archive)
except ImportError:
    pass
